# CryoET–SCSI on MNIST — reconstruction grid

Load a trained checkpoint and render a **`[clean | FBP | recon]`** comparison grid.

- **clean** — the ground-truth MNIST digit
- **FBP** — the classical filtered-back-projection baseline (also the SCSI warmup target)
- **recon** — the learned reconstruction (transport the forward-model prior through the EMA drift)

> The unknown global rotation θ₀ is **not observable**, so reconstructions are correct only *up to a rotation*. The last cell shows a rotation-aligned comparison and reports MSE-up-to-rotation.

In [ ]:
import os, sys, math, json

# This notebook lives in src_clean/. Make sure its modules are importable
# regardless of where Jupyter was launched from.
if "forward.py" not in os.listdir(os.getcwd()) and os.path.isdir("src_clean"):
    os.chdir("src_clean")
sys.path.insert(0, os.getcwd())

import torch
import numpy as np
import matplotlib.pyplot as plt

from data import load_mnist_pm1
from forward import radon_tilt_series, rotate_image
from backwards import warmup_target
from model import build_model
from scsi import SCSInterpolant

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# --- config -------------------------------------------------------------
# Point CKPT at a checkpoint saved by main.py (model-best.pt / model-latest.pt).
# Env vars allow non-interactive execution; edit the defaults for interactive use.
CKPT      = os.environ.get("SCSI_CKPT", "./results_clean/model-best.pt")
DATA_ROOT = os.environ.get("SCSI_DATA_ROOT", "./data")
N_IMAGES  = int(os.environ.get("SCSI_N", "8"))
SEED      = int(os.environ.get("SCSI_SEED", "0"))

# Pull the exact forward-model + network hyperparameters from the run's args.json
# (written next to the checkpoint by main.py) so they match the trained net.
hp = dict(K=16, tilt_span_deg=60.0, epsilon=0.0, ode_steps=80, model_channels=32)
args_path = os.path.join(os.path.dirname(CKPT), "args.json")
if os.path.exists(args_path):
    with open(args_path) as f:
        saved = json.load(f)
    hp.update({k: saved.get(k, hp[k]) for k in hp})
    print("loaded hyperparameters from", args_path)
else:
    print("no args.json next to checkpoint; using defaults below — edit if they "
          "do not match the trained model (K / model_channels MUST match).")

K, TILT_SPAN, EPSILON = hp["K"], hp["tilt_span_deg"], hp["epsilon"]
ODE_STEPS, MODEL_CH   = hp["ode_steps"], hp["model_channels"]
print(hp)

In [ ]:
# --- pick test images, corrupt them, and compute the FBP baseline -------
g = torch.Generator().manual_seed(SEED)
base = load_mnist_pm1(DATA_ROOT)
idx = torch.randint(len(base), (N_IMAGES,), generator=g).tolist()
clean = torch.stack([base[i][0] for i in idx]).to(device)        # [N,1,32,32]

fwd = radon_tilt_series(EPSILON, K, TILT_SPAN)
cgen = torch.Generator().manual_seed(SEED + 1)
z_out, cond = fwd(clean, return_latents=True, generator=cgen)    # z_out: FM prior, cond: projections
z_out, cond = z_out.to(device), cond.to(device)

fbp_img = warmup_target(cond, K, TILT_SPAN)                       # [N,1,32,32]
print("clean", tuple(clean.shape), "cond", tuple(cond.shape), "fbp", tuple(fbp_img.shape))

In [ ]:
# --- load the checkpoint (EMA weights) and reconstruct ------------------
model = build_model(D=32, nc=1, K=K, model_channels=MODEL_CH).to(device)
ckpt = torch.load(CKPT, map_location=device)
model.load_state_dict(ckpt["ema"])                               # EMA weights
model.eval()

interp = SCSInterpolant(fwd, n_steps=ODE_STEPS)
with torch.no_grad():
    recon = interp.transport(model, z_out, cond)                 # noise -> clean
print("reconstructed at training step", ckpt.get("step"), "| recon", tuple(recon.shape))

In [ ]:
# --- metrics + best-rotation alignment ----------------------------------
@torch.no_grad()
def best_rotation(recon, clean, n_angles=72):
    """Per image: min MSE over a grid of global rotations + the aligned recon."""
    N = recon.shape[0]
    best_mse = torch.full((N,), float("inf"), device=recon.device)
    best_img = recon.clone()
    for a in torch.linspace(0, 2 * math.pi, n_angles + 1)[:-1]:
        ang = torch.full((N,), float(a), device=recon.device)
        r = rotate_image(recon, ang, bg=-1.0)
        m = ((r - clean) ** 2).flatten(1).mean(1)
        upd = m < best_mse
        best_mse = torch.minimum(best_mse, m)
        best_img[upd] = r[upd]
    return best_mse, best_img

mse     = ((recon - clean) ** 2).flatten(1).mean(1)
fbp_mse = ((fbp_img - clean) ** 2).flatten(1).mean(1)
mse_rot, recon_aligned = best_rotation(recon, clean)
print(f"recon MSE                 : {mse.mean():.4f}")
print(f"recon MSE up-to-rotation  : {mse_rot.mean():.4f}")
print(f"FBP baseline MSE          : {fbp_mse.mean():.4f}")

In [ ]:
# --- grid plotting helper -----------------------------------------------
def show_grid(rows, title):
    n = rows[0][1].shape[0]
    fig, axes = plt.subplots(len(rows), n, figsize=(1.4 * n, 1.5 * len(rows)))
    axes = np.array(axes).reshape(len(rows), n)
    for r, (label, imgs) in enumerate(rows):
        imgs = imgs.detach().cpu()
        for c in range(n):
            ax = axes[r, c]
            ax.imshow(imgs[c, 0], cmap="gray", vmin=-1, vmax=1)
            ax.set_xticks([]); ax.set_yticks([])
            if c == 0:
                ax.set_ylabel(label, fontsize=11)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

show_grid([("clean", clean), ("FBP", fbp_img), ("recon", recon)],
          "CryoET–SCSI on MNIST  ·  [clean | FBP | recon]")

### Rotation-aligned view

Because the global orientation θ₀ is unobservable, the raw `recon` row above may appear rotated relative to `clean`. Below, each reconstruction is rotated to its best-matching angle for a fair visual comparison (this is the `MSE up-to-rotation` reported above).

In [ ]:
show_grid([("clean", clean), ("recon", recon), ("recon (best-rot)", recon_aligned)],
          "Rotation-aligned reconstruction")